<a href="https://colab.research.google.com/github/LeMaterial/lematerial-llm-synthesis/blob/main/examples/notebooks/tutorials/03_batch_extraction_with_the_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Tutorial 3 — Batch extraction with the `lemat-synth` CLI

The shortest path from a stack of papers to structured data: one command, with
concurrency, resumability and a documented config file. This is how most people
run the pipeline, so it comes before the guided tour — Tutorial 4 takes the same
pipeline apart stage by stage, which is what you want when you need to change
what it does rather than just run it.

## What you'll learn

1. The two commands: `extract` (one paper) and `batch` (a folder)
2. How `config/cli.yaml` works and how to override any value from the command
   line with Hydra `key=value` syntax
3. How to point individual pipeline components at different models — and
   different API keys
4. What the output directory looks like, and how to load it back into pandas

## Prerequisites

- The package installed (`uv sync && uv pip install -e .`)
- **Runtime:** ~5 min for the toy example below. **Cost:** a fraction of a cent
  with the default Gemini models.

> The CLI finds `config/cli.yaml` and `.env` relative to the **installed
> package**, not your working directory, so these commands work from anywhere —
> including this notebook's folder.

## Setup — local or Colab

This notebook runs unchanged in two places:

- **Locally**, from a clone of the repository (`uv sync && uv pip install -e .`),
  with your API keys in the `.env` file at the repository root.
- **On [Google Colab](https://colab.research.google.com)** — click the badge at
  the top. The cell below clones the repository and installs it, which takes a
  few minutes the first time, then reads your keys from Colab's **secret
  manager**: open the 🔑 icon in the left sidebar, add one secret per key
  (`GEMINI_API_KEY`, `HF_TOKEN`, …) and switch *Notebook access* on for each.

Either way the keys land in `os.environ` and nothing else in the notebook
changes — no key is ever passed as a function argument, so none of them can end
up in the notebook's output or in git.

> If an import fails immediately after the setup cell on Colab, use
> **Runtime → Restart session** and run it again: the clone is cached, so the
> second run is quick.


In [ ]:
# --- Setup: this cell is the only difference between local and Colab -----
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Every key the project knows about. A tutorial only needs a subset; whichever
# ones are missing are reported by the key check further down.
API_KEY_NAMES = (
    "GEMINI_API_KEY",
    "ANTHROPIC_API_KEY",
    "MISTRAL_API_KEY",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
    "HF_TOKEN",
)

if IN_COLAB:
    REPO_URL = "https://github.com/LeMaterial/lematerial-llm-synthesis.git"
    # The installed code has to match this notebook: config/cli.yaml's default
    # models and the pipeline are read from the clone, so a stale branch means
    # overrides in the cells below refer to models the config does not use.
    # Point this at your own branch when running an unmerged tutorial.
    REPO_BRANCH = "main"
    REPO_ROOT = Path("/content/lematerial-llm-synthesis")

    if not REPO_ROOT.exists():
        print(f"Cloning the repository ({REPO_BRANCH}) ...")
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1",
             "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
            check=True,
        )
    print("Installing llm-synthesis (a few minutes on the first run) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO_ROOT)],
        check=True,
    )

    # Colab keeps secrets outside the notebook, so they cannot leak into its
    # output: add them under the key icon in the left sidebar.
    from google.colab import userdata

    for name in API_KEY_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # not set, or notebook access not granted - reported below
    KEY_SOURCE = "Colab secrets"
else:
    from dotenv import find_dotenv, load_dotenv

    def find_repo_root(start: Path | None = None) -> Path:
        """Walk up from `start` (default: cwd) until a directory has pyproject.toml."""
        here = (start or Path.cwd()).resolve()
        for candidate in (here, *here.parents):
            if (candidate / "pyproject.toml").exists():
                return candidate
        raise RuntimeError(f"No pyproject.toml found above {here}")

    REPO_ROOT = find_repo_root()
    # find_dotenv walks up from the working directory, so this works whether you
    # started Jupyter at the repo root or inside this folder.
    env_path = find_dotenv(usecwd=True)
    load_dotenv(env_path, override=True)
    KEY_SOURCE = env_path or "no .env found"

print(f"environment: {'Google Colab' if IN_COLAB else 'local'}")
print(f"repo root:   {REPO_ROOT}")
print(f"API keys:    {KEY_SOURCE}")


## Step 0a — Choose your provider

Two ways to reach the models, chosen with a single flag:

- **Direct** (default) — one key per provider (`GEMINI_API_KEY`,
  `ANTHROPIC_API_KEY`, `MISTRAL_API_KEY`). No overrides needed:
  `config/cli.yaml`'s defaults already point at Gemini and Claude.
- **OpenRouter** — one `OPENROUTER_API_KEY` for everything, and any model id
  from [openrouter.ai/models](https://openrouter.ai/models). Useful if you
  would rather not hold accounts with three providers, or want an open-weight
  model.

Set `USE_OPENROUTER = True` below and every `lemat-synth extract`/`batch` call
in this notebook picks it up automatically through the `MODEL_ARGS` variable —
no other cell changes needed.

In [ ]:
import os

# ==============================================================================
# USER CONFIGURATION
# ==============================================================================

USE_OPENROUTER = True  # True routes every lemat-synth model through OpenRouter

OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"

# Model per pipeline role. Only used when USE_OPENROUTER = True - direct-API
# defaults come from config/cli.yaml and don't need to be repeated here.
OPENROUTER_MODELS = {
    "synthesis": "google/gemini-3.5-flash-lite",
    "material": "google/gemini-3.5-flash-lite",
    "judge": "google/gemini-3.5-flash-lite",
    "plot": "anthropic/claude-sonnet-4.6",  # only used with with_performance=true
}


def model_overrides(synthesis_model: str | None = None) -> str:
    """Hydra `key=value` overrides for `lemat-synth`, honouring USE_OPENROUTER.

    Direct APIs need no overrides - config/cli.yaml's defaults already apply.

    `api_base` is a single global setting that every component model goes
    through, so on the OpenRouter path material and judge have to be overridden
    together with synthesis - otherwise config/cli.yaml's `gemini/...` defaults
    get sent to OpenRouter's URL and come back 404. Pass `synthesis_model` to
    change that one slot while keeping the rest consistent.
    """
    if not USE_OPENROUTER:
        return f"synthesis_model={synthesis_model}" if synthesis_model else ""
    synthesis = (
        synthesis_model or f"openrouter/{OPENROUTER_MODELS['synthesis']}"
    )
    return (
        f"synthesis_model={synthesis} "
        f"material_model=openrouter/{OPENROUTER_MODELS['material']} "
        f"judge_model=openrouter/{OPENROUTER_MODELS['judge']} "
        f"api_base={OPENROUTER_API_BASE}"
    )


MODEL_ARGS = model_overrides()
print(f"provider: {'OpenRouter' if USE_OPENROUTER else 'direct APIs'}")
print(f"overrides: {MODEL_ARGS or '(none - using config/cli.yaml defaults)'}")

## Step 0b — API keys and your `.env` file

Nothing in this project takes an API key as a function argument. Keys live in a
single `.env` file at the repository root, get loaded into the process
environment once per session, and LiteLLM/DSPy read them from there. That means
**your keys never appear in notebook code, notebook outputs, or git history**.

### Create your `.env`

From the repository root:

```bash
cp .env.example .env
```

Then open `.env` and fill in the keys you need — one per line, no quotes and no
spaces around `=`:

```
GEMINI_API_KEY=AIza...
ANTHROPIC_API_KEY=sk-ant-...
```

`.env` is git-ignored, so it never gets committed.

### Keys used by this tutorial

Which keys you need depends on the `USE_OPENROUTER` flag set in Step 0a.

**Direct providers (`USE_OPENROUTER = False`, the default):**

| Key | What it unlocks | Needed here? |
|-----|-----------------|---------------|
| `GEMINI_API_KEY` | The default `synthesis_model`, `material_model` and `judge_model` | **yes** |
| `ANTHROPIC_API_KEY` | The default `plot_model`, used when `with_performance=true` | only with performance |
| `MISTRAL_API_KEY` | `pdf_extractor=mistral`; the default `docling` needs no key | no |

**OpenRouter (`USE_OPENROUTER = True`):**

| Key | What it unlocks | Needed here? |
|-----|-----------------|---------------|
| `OPENROUTER_API_KEY` | Every model in `OPENROUTER_MODELS` — synthesis, material, judge and (with performance) plot | **yes** |
| `MISTRAL_API_KEY` | `pdf_extractor=mistral`; OCR does not go through OpenRouter | no |

Where to get them: **Gemini** (free tier is enough for this tutorial) at
[aistudio.google.com](https://aistudio.google.com/app/apikey), **Anthropic** at
[console.anthropic.com](https://console.anthropic.com/), **Mistral** at
[console.mistral.ai](https://console.mistral.ai/), **OpenRouter** at
[openrouter.ai/keys](https://openrouter.ai/keys).

> The CLI calls `load_dotenv()` on the repository-root `.env` itself before it
> does anything else (`_load_env()` in `src/llm_synthesis/cli.py`), so you do
> **not** have to export anything into your shell. Editing `.env` is enough.

The next cell loads `.env` and reports which keys arrived — it prints only the
key *length*, never the value, so the output is safe to share.

> **On Colab you do not need a `.env` file.** The setup cell above already
> read your keys from Colab's secret manager — add them there instead (🔑 in
> the left sidebar), with *Notebook access* switched on.


In [ ]:
if USE_OPENROUTER:
    REQUIRED_KEYS = {"OPENROUTER_API_KEY": "every default model in this tutorial"}
    OPTIONAL_KEYS = {"MISTRAL_API_KEY": "only with pdf_extractor=mistral"}
else:
    REQUIRED_KEYS = {"GEMINI_API_KEY": "default synthesis/material/judge models"}
    OPTIONAL_KEYS = {
        "ANTHROPIC_API_KEY": "plot extraction, only with with_performance=true",
        "MISTRAL_API_KEY": "only with pdf_extractor=mistral",
    }


def report_keys(required, optional):
    """Print which API keys the environment provided, never their values."""
    print(f"API keys from: {KEY_SOURCE}")
    print(f"provider: {'OpenRouter' if USE_OPENROUTER else 'direct APIs'}\n")
    missing = []
    for name, purpose in {**required, **optional}.items():
        value = os.getenv(name)
        is_required = name in required
        if value:
            status = f"set ({len(value)} chars)"
        elif is_required:
            status = "MISSING"
            missing.append(name)
        else:
            status = "not set"
        tag = "required" if is_required else "optional"
        print(f"  {name:<28} {status:<16} [{tag}] {purpose}")
    if missing:
        fix = (
            "Add them in Colab's secret manager (the key icon in the left "
            "sidebar) and switch on notebook access, then re-run this cell."
            if IN_COLAB
            else "Copy .env.example to .env at the repository root and fill "
            "them in, then re-run this cell."
        )
        raise RuntimeError(
            "Missing required key(s): " + ", ".join(missing) + ". " + fix
        )
    print("\nAll required keys are present.")


report_keys(REQUIRED_KEYS, OPTIONAL_KEYS)

## Step 1 — The two commands

`extract` takes one file, `batch` takes a folder. Both accept `.txt`, `.md` and
`.pdf` inputs, and both take Hydra-style `key=value` overrides *after* the path.

In [ ]:
!lemat-synth --help

In [ ]:
!lemat-synth extract --help

## Step 2 — A paper to work on

No sample papers ship with this repo (`data/` is git-ignored), so this cell
writes a **small synthetic paper** — a few paragraphs invented for this
tutorial, not a real publication — just so the commands below have something to
chew on. Replace `PAPERS_DIR` with a folder of your own papers when you move on
to real work.

In [ ]:
# Everything this notebook writes goes under data/, which is git-ignored -
# demo inputs and CLI results should never end up in a commit.
DEMO_DIR = REPO_ROOT / "data" / "tutorials" / "cli_demo"
PAPERS_DIR = DEMO_DIR / "papers"
OUTPUT_DIR = DEMO_DIR / "results"
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

demo_paper = """# Hydrothermal synthesis of Co-doped ZnO nanorods

*This is a synthetic example written for the LeMat-Synth tutorials. It is not a
real publication and the numbers in it are invented.*

## Experimental

Zn(NO3)2 - 6H2O (2.97 g, 99.9%, Sigma-Aldrich) and Co(NO3)2 - 6H2O (0.29 g)
were dissolved in 80 mL of deionised water under magnetic stirring for 30 min
at room temperature. Hexamethylenetetramine (1.40 g) was then added and the
solution was stirred for a further 15 min.

The mixture was transferred into a 100 mL Teflon-lined stainless steel
autoclave and held at 95 C for 6 h. The autoclave was then allowed to cool
naturally to room temperature.

The white precipitate was collected by centrifugation at 6000 rpm, washed three
times with deionised water and once with ethanol, and dried in a vacuum oven at
60 C for 12 h. The resulting powder was finally calcined in air at 450 C for
2 h with a heating rate of 5 C/min to give Co-doped ZnO (Zn0.95Co0.05O)
nanorods.

An undoped ZnO reference sample was prepared by the identical route, omitting
the cobalt precursor.
"""

PAPER_PATH = PAPERS_DIR / "co_doped_zno_demo.md"
PAPER_PATH.write_text(demo_paper)

print(f"Wrote {PAPER_PATH} ({len(demo_paper)} characters)")

## Step 3 — Extract a single paper

The defaults live in `config/cli.yaml`. Everything you see there can be
overridden on the command line; here we only redirect the output.

In [ ]:
print(f"Executing command:\n\n  lemat-synth extract {PAPER_PATH} \\ \n\toutput_dir={OUTPUT_DIR} \\ \n\t{MODEL_ARGS}")

In [ ]:
!lemat-synth extract {PAPER_PATH} output_dir={OUTPUT_DIR} {MODEL_ARGS}

### Overriding configuration

Any key in `config/cli.yaml` can be set with `key=value`, including nested ones
using dots. A few that matter in practice:

| Override | Effect |
|----------|--------|
| `synthesis_model=anthropic/claude-sonnet-4-6` | Any LiteLLM model string works |
| `domain=catalysis` | Plot-relevance filter: `generic`, `catalysis`, `superconductors`, `electrochemistry` |
| `with_performance=true` | Also digitise plots and link them to materials |
| `pdf_extractor=mistral` | Use Mistral OCR instead of local Docling for PDFs |
| `figure_segmenter=florence` | Florence-2 instead of Grounding DINO + ResNet |
| `"prompts.synthesis_instructions='...'"` | Rewrite any prompt without touching code |

**NOTE**: For overriding prompts, use quotes around the whole `key=value` pair
and around the value, like: `"key='value'"`.

**`api_base` is global.** It is one value shared by every component model, not
a per-model setting — so pointing it at OpenRouter while overriding only
`synthesis_model` sends the *default* `gemini/...` material and judge models to
OpenRouter's URL, which answers `404 Not Found`. Change the provider for all
slots at once, which is what `model_overrides()` from Step 0a does.

To switch *every* model at once — rather than overriding one — use the
`USE_OPENROUTER` flag from Step 0a; it already threads through `MODEL_ARGS`
into every `lemat-synth` call in this notebook.

In [ ]:
# Same paper, a different synthesis model and a custom instruction.
# Cheap models are fine for a short synthetic paper like this one.
#
# Only the synthesis slot changes here - model_overrides() keeps material and
# judge on the same provider, because api_base applies to all three at once.
ALT_SYNTHESIS_MODEL = (
    "openrouter/google/gemini-3.5-flash"
    if USE_OPENROUTER
    else "gemini/gemini-3-flash-lite"
)
custom_model_overrides = model_overrides(ALT_SYNTHESIS_MODEL)
print(f"overrides: {custom_model_overrides}")

!lemat-synth extract {PAPER_PATH} \
    output_dir={DEMO_DIR / "results_flash"} \
    {custom_model_overrides} \
    "prompts.synthesis_instructions='Extract the synthesis procedure. Preserve every quantitative detail, including concentrations, ramp rates and washing steps.'"

### Per-slot OpenRouter keys

`USE_OPENROUTER = True` in Step 0a already routes every command in this
notebook through OpenRouter with a single `OPENROUTER_API_KEY`. Skip this
section unless you want **per-model** keys instead — e.g. separate OpenRouter
accounts for different models:

```bash
lemat-synth extract paper.md \
    synthesis_model=openrouter/qwen/qwen3.5-397b-a17b \
    synthesis_api_key_env=OPENROUTER_QWEN_API_KEY \
    api_base=https://openrouter.ai/api/v1
```

`synthesis_api_key_env` (and `material_api_key_env`, `judge_api_key_env`)
only accept the names in `_ALLOWED_API_KEY_ENVS` (`src/llm_synthesis/cli.py`) —
`ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY`, `MISTRAL_API_KEY`,
`OPENROUTER_QWEN_API_KEY`, `OPENROUTER_KIMI_API_KEY`,
`OPENROUTER_DEEPSEEK_API_KEY` — and a generic `OPENROUTER_API_KEY` is
deliberately not among them, so a typo cannot silently fall back to
auto-detection. Leave `*_api_key_env` at `null` (the default) to let LiteLLM
auto-detect `OPENROUTER_API_KEY` instead.

One caveat for `with_performance=true`: the plot VLM does **not** go through
LiteLLM. `ClaudeAPIClient` builds an Anthropic client directly, and on the
`openrouter/` path it sets the base URL but reads the key from
`ANTHROPIC_API_KEY`. So to route plots through OpenRouter, set
`plot_model=openrouter/anthropic/claude-sonnet-4.6` **and** put your
OpenRouter key in `ANTHROPIC_API_KEY` (or, for this session only,
`os.environ["ANTHROPIC_API_KEY"] = os.environ["OPENROUTER_API_KEY"]`).

## Step 4 — Batch a folder

`batch` walks a directory, detects `_SI` companion files, processes several
papers concurrently and skips papers that already have results — so an
interrupted run resumes where it stopped.

| Key | Default | Meaning |
|-----|---------|---------|
| `max_papers` | `null` | Stop after N papers |
| `skip_existing` | `true` | Skip papers with an existing results folder |
| `max_papers_parallel` | `4` | Papers in flight at once |

LLM concurrency *within* a paper is capped separately by the
`LLM_SYNTHESIS_MAX_CONCURRENT_LLM_CALLS` environment variable (default 10) —
lower it if your provider starts returning rate-limit errors.

In [ ]:
!lemat-synth batch {PAPERS_DIR} output_dir={DEMO_DIR / "results_batch"} max_papers=2 max_papers_parallel=2 {MODEL_ARGS}

## Step 5 — Read the results back

The output layout is one directory per paper:

```
results/cli_batch/
└── <paper_id>/
    ├── <material>.json          one file per extracted material
    ├── performance_mappings.json  plot ↔ material links (empty without --with_performance)
    └── linking_summary.json       per-paper summary and linking statistics
```

Each `<material>.json` holds the material name, the full synthesis ontology, the
judge's evaluation, and — when performance extraction ran — the linked plot
data.

In [ ]:
import json

for paper_dir in sorted(p for p in OUTPUT_DIR.iterdir() if p.is_dir()):
    print(f"{paper_dir.name}/")
    for f in sorted(paper_dir.iterdir()):
        print(f"    {f.name:<40} {f.stat().st_size:>8,} bytes")

In [ ]:
summary_paths = list(OUTPUT_DIR.glob("*/linking_summary.json"))
if summary_paths:
    print(json.dumps(json.loads(summary_paths[0].read_text()), indent=2)[:1200])
else:
    print("No linking_summary.json found - did the extract cell above succeed?")

In [ ]:
import pandas as pd

records = []
for material_file in OUTPUT_DIR.glob("*/*.json"):
    if material_file.name in {
        "linking_summary.json",
        "performance_mappings.json",
    }:
        continue
    entry = json.loads(material_file.read_text())
    synthesis = entry.get("synthesis") or {}
    evaluation = entry.get("evaluation") or {}
    scores = evaluation.get("scores") or {}
    records.append(
        {
            "paper": material_file.parent.name,
            "material": entry.get("material"),
            "method": synthesis.get("synthesis_method"),
            "type": synthesis.get("target_compound_type"),
            "n_steps": len(synthesis.get("steps") or []),
            "n_precursors": len(synthesis.get("starting_materials") or []),
            "judge_overall": scores.get("overall_score"),
        }
    )

df = pd.DataFrame(records)
print(f"{len(df)} material record(s)")
df

In [ ]:
# The full recipe for the first material, as the pipeline wrote it.
if records:
    first = sorted(
        f
        for f in OUTPUT_DIR.glob("*/*.json")
        if f.name not in {"linking_summary.json", "performance_mappings.json"}
    )[0]
    entry = json.loads(first.read_text())
    print(json.dumps(entry.get("synthesis"), indent=2)[:2500])

## Step 6 — When to use the Python API instead

The CLI is the right tool for "run the standard pipeline over these papers".
Reach for the Python API when you need to change *what the pipeline is*:

```python
from llm_synthesis.services.pipelines.synthesis_performance_pipeline import (
    SynthesisPerformancePipeline,
)

pipeline = SynthesisPerformancePipeline(
    material_extractor=...,     # any MaterialExtractorInterface
    synthesis_extractor=...,    # any SynthesisExtractorInterface
    judge=...,                  # optional
    ...
)
result = pipeline.process_paper(paper)
```

Both `process_paper` (sync) and `process_paper_async` (asyncio + semaphore)
exist; the CLI uses the async one. Tutorial 4 builds these components
explicitly, and the Hydra deployment scripts in
`examples/scripts/deployment/` show the same pipeline wired up from YAML config
groups for large runs.

## What's next

- **[Tutorial 4 — Synthesis + performance from a paper](04_extracting_synthesis_and_performance.ipynb)**:
  what `with_performance=true` actually does under the hood, one stage at a time.
- **[Tutorial 5 — Evaluating extraction quality](05_evaluating_extraction_quality.ipynb)**:
  the `judge_overall` column you just loaded, and whether you should believe it.
- **[Tutorial 6 — Customising the ontology](06_customizing_the_ontology.ipynb)**:
  change what gets extracted in the first place.

Clean-up: everything this notebook wrote lives under
`data/tutorials/cli_demo/`, which is git-ignored — delete it whenever you like.
